In [8]:
%env WORKDIR=/tmp/vault      

env: WORKDIR=/tmp/vault


In [9]:
import os
from dotenv import load_dotenv

load_dotenv("./.env")

VAULT_TOKEN = os.getenv('VAULT_TOKEN')
VAULT_ADDR = os.getenv('VAULT_ADDR')
VAULT_CACERT = os.getenv('VAULT_CACERT')
AWS_REGION = os.getenv('AWS_REGION')

In [10]:
! vault status

Key                      Value
---                      -----
Seal Type                awskms
Recovery Seal Type       shamir
Initialized              true
Sealed                   false
Total Recovery Shares    1
Threshold                1
Version                  2.0.3+ent
Build Date               2026-06-16T21:32:56Z
Storage Type             raft
Cluster Name             vault-cluster-e89137a9
Cluster ID               b24418e0-d91d-d6d3-c571-5af49b1ec963
Removed From Cluster     false
HA Enabled               true
HA Cluster               https://vault-0.vault-internal:8201
HA Mode                  active
Active Since             2026-07-23T13:27:14.354277171Z
Raft Committed Index     8821
Raft Applied Index       8821
Last WAL                 3408


In [11]:
import csv
import os
from pathlib import Path

credentials_file = Path(".vault/vault_test_accessKeys.csv")
if not credentials_file.is_file():
    raise FileNotFoundError(f"No se encuentra el fichero de credenciales: {credentials_file}")

# utf-8-sig elimina el BOM que algunos exportadores incluyen en la primera cabecera.
with credentials_file.open(newline="", encoding="utf-8-sig") as csv_file:
    credentials = next(csv.DictReader(csv_file), None)

required_columns = ("Access key ID", "Secret access key")
if not credentials or any(not credentials.get(column, "").strip() for column in required_columns):
    raise ValueError(f"El CSV debe contener valores para: {', '.join(required_columns)}")

os.environ["AWS_ACCESS_KEY_ID"] = credentials["Access key ID"].strip()
os.environ["AWS_SECRET_ACCESS_KEY"] = credentials["Secret access key"].strip()
os.environ["TF_VAR_aws_access_key_id"] = os.environ["AWS_ACCESS_KEY_ID"]
os.environ["TF_VAR_aws_secret_access_key"] = os.environ["AWS_SECRET_ACCESS_KEY"]
# Evita mezclar estas claves con un token temporal de una ejecución anterior de Doormat.
os.environ.pop("AWS_SESSION_TOKEN", None)

print(f"Credenciales AWS cargadas desde {credentials_file} (sin mostrar valores).")

Credenciales AWS cargadas desde .vault/vault_test_accessKeys.csv (sin mostrar valores).


In [12]:
! vault write -f sys/activation-flags/secrets-sync/activate

Key            Value
---            -----
activated      [secrets-sync]
unactivated    [oauth-resource-server secrets-import]


In [ ]:
! terraform -chdir=terraform-static validate

Success! The configuration is valid.



In [ ]:
!  terraform -chdir=terraform-static apply -auto-approve

vault_secrets_sync_config.global_config: Refreshing state... [id=global_config]
data.aws_region.current: Reading...
aws_iam_openid_connect_provider.vault_secrets_sync: Refreshing state... [id=arn:aws:iam::220065406052:oidc-provider/vault.jose-merchan.sbx.hashidemos.io/v1/identity/oidc/secrets-sync]
data.aws_region.current: Read complete after 0s [id=eu-central-1]
aws_iam_policy.secret_sync: Refreshing state... [id=arn:aws:iam::220065406052:policy/vault-secret-sync]
aws_iam_role.secret_sync: Refreshing state... [id=vault-secret-sync]
aws_iam_role_policy_attachment.secret_sync: Refreshing state... [id=vault-secret-sync/arn:aws:iam::220065406052:policy/vault-secret-sync]

Terraform used the selected providers to generate the following execution plan.
Resource actions are indicated with the following symbols:
  + create

Terraform will perform the following actions:

  # vault_secrets_sync_aws_destination.aws will be created
  + resource "vault_secrets_sync_aws_destination" "aws" {
      +

In [1]:
! vault read sys/sync/destinations/aws-sm/aws-sm-dest

Key                   Value
---                   -----
connection_details    map[access_key_id:***** region:eu-central-1 secret_access_key:*****]
name                  aws-sm-dest
options               map[custom_tags:map[Managed_by:HashiCorp Vault] granularity_level:secret-path secret_name_template:vault_sync_{{ .SecretBaseName | lowercase }}]
type                  aws-sm


In [2]:
! vault kv patch -mount=sync-aws-static verification sync_retry="$(date -u +%FT%TZ)"

== Secret Path ==
kv/data/test

======= Metadata =======
Key                Value
---                -----
created_time       2026-07-23T13:58:07.97899901Z
custom_metadata    <nil>
deletion_time      n/a
destroyed          false
version            3


In [3]:
! vault read -format=json \
  sys/sync/destinations/aws-sm/aws-sm-dest/associations

{
  "request_id": "173fd2cf-bf79-b58d-7380-894cb3340096",
  "lease_id": "",
  "lease_duration": 0,
  "renewable": false,
  "data": {
    "associated_secrets": {
      "kv_49ccfaec/test": {
        "accessor": "kv_49ccfaec",
        "external_name": "vault_sync_test",
        "last_operation": "Write",
        "mount": "kv",
        "secret_name": "test",
        "sync_status": "SYNCED",
        "updated_at": "2026-07-23T13:58:08.213392615Z"
      }
    },
    "store_name": "aws-sm-dest",
    "store_type": "aws-sm",
    "sync_operation_counters": {
      "SYNCED": 1
    }
  },
  "warnings": null,
  "mount_type": "system"
}


In [ ]:
%%bash
## CLEAN UP

terraform -chdir=terraform-static destroy -auto-approve

data.aws_region.current: Reading...
data.aws_region.current: Read complete after 0s [id=eu-central-1]
vault_secrets_sync_aws_destination.aws: Refreshing state... [id=aws-sm-dest]

Terraform used the selected providers to generate the following execution
plan. Resource actions are indicated with the following symbols:
  - destroy

Terraform will perform the following actions:

  # vault_secrets_sync_aws_destination.aws will be destroyed
  - resource "vault_secrets_sync_aws_destination" "aws" {
      - access_key_id              = (sensitive value) -> null
      - custom_tags                = {
          - "Managed_by" = "HashiCorp Vault"
        } -> null
      - disable_strict_networking  = false -> null
      - granularity                = "secret-path" -> null
      - id                         = "aws-sm-dest" -> null
      - identity_token_audience_wo = (write-only attribute) -> null
      - identity_token_key_wo      = (write-only attribute) -> null
      - name                    